In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("lab2-clickhouse")
    .getOrCreate()
)

pg_url = "jdbc:postgresql://postgres_db:5432/bigdata_warehouse"
pg_props = {
    "user": "warehouse_user",
    "password": "secure_pass_2026",
    "driver": "org.postgresql.Driver"
}

ch_url = "jdbc:clickhouse://clickhouse-server/warehouse"
ch_props = {
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
    "user": "warehouse",
    "password": "ch_pass_2026"
}

In [3]:
from pyspark.sql.functions import sum, avg, count, desc

In [ ]:
fact_sales = spark.read.jdbc(pg_url, "fact_sales", properties=pg_props)
dim_product = spark.read.jdbc(pg_url, "dim_product", properties=pg_props)
dim_customer = spark.read.jdbc(pg_url, "dim_customer", properties=pg_props)
dim_store = spark.read.jdbc(pg_url, "dim_store", properties=pg_props)
dim_supplier = spark.read.jdbc(pg_url, "dim_supplier", properties=pg_props)

1. Витрина по продуктам

In [5]:
product_sales = (
    fact_sales.alias("f")
    .join(
        dim_product.alias("p"),
        col("f.product_id") == col("p.id"),
        "inner"
    )
)

mart_product_sales = (
    product_sales
    .groupBy(
        col("p.id").alias("product_id"),
        col("p.product_name"),
        col("p.product_category")
    )
    .agg(
        sum("f.sale_quantity").alias("total_sales_qty"),
        sum("f.sale_total_price").alias("total_revenue"),
        avg("p.product_rating").alias("avg_rating"),
        avg("p.product_reviews").alias("avg_reviews"),
        count("f.id").alias("sales_count")
    )
    .orderBy(desc("total_revenue"))
)

2. Витрина по клиентам

In [6]:
customer_sales = (
    fact_sales.alias("f")
    .join(
        dim_customer.alias("c"),
        col("f.customer_id") == col("c.id"),
        "inner"
    )
)

mart_customer_sales = (
    customer_sales
    .groupBy(
        col("c.id").alias("customer_id"),
        col("c.first_name"),
        col("c.last_name"),
        col("c.country")
    )
    .agg(
        sum("f.sale_total_price").alias("total_spent"),
        avg("f.sale_total_price").alias("avg_check"),
        count("f.id").alias("orders_count")
    )
    .orderBy(desc("total_spent"))
)

3. Витрина по времени

In [7]:
from pyspark.sql.functions import to_date, year, month

mart_time_sales = (
    fact_sales
    .withColumn("sale_dt", to_date(col("sale_date")))
    .groupBy(
        year("sale_dt").alias("year"),
        month("sale_dt").alias("month")
    )
    .agg(
        sum("sale_total_price").alias("total_revenue"),
        sum("sale_quantity").alias("total_sales_qty"),
        avg("sale_total_price").alias("avg_check"),
        count("id").alias("orders_count")
    )
    .orderBy("year", "month")
)

4. Витрина по магазинам

In [8]:
store_sales = (
    fact_sales.alias("f")
    .join(
        dim_product.alias("p"),
        col("f.product_id") == col("p.id"),
        "inner"
    )
    .join(
        dim_store.alias("st"),
        col("p.store_id") == col("st.id"),
        "inner"
    )
)

mart_store_sales = (
    store_sales
    .groupBy(
        col("st.id").alias("store_id"),
        col("st.name"),
        col("st.city"),
        col("st.country")
    )
    .agg(
        sum("f.sale_total_price").alias("total_revenue"),
        avg("f.sale_total_price").alias("avg_check"),
        count("f.id").alias("orders_count")
    )
    .orderBy(desc("total_revenue"))
)

5. Витрина по поставщикам

In [9]:
supplier_sales = (
    fact_sales.alias("f")
    .join(
        dim_product.alias("p"),
        col("f.product_id") == col("p.id"),
        "inner"
    )
    .join(
        dim_supplier.alias("sp"),
        col("p.supplier_id") == col("sp.id"),
        "inner"
    )
)

mart_supplier_sales = (
    supplier_sales
    .groupBy(
        col("sp.id").alias("supplier_id"),
        col("sp.name"),
        col("sp.country")
    )
    .agg(
        sum("f.sale_total_price").alias("total_revenue"),
        avg("p.product_price").alias("avg_product_price"),
        count("f.id").alias("orders_count")
    )
    .orderBy(desc("total_revenue"))
)

6. Витрина качества

In [10]:
mart_product_quality = (
    fact_sales.alias("f")
    .join(
        dim_product.alias("p"),
        col("f.product_id") == col("p.id"),
        "inner"
    )
    .groupBy(
        col("p.id").alias("product_id"),
        col("p.product_name")
    )
    .agg(
        avg("p.product_rating").alias("avg_rating"),
        avg("p.product_reviews").alias("avg_reviews"),
        sum("f.sale_quantity").alias("total_sales_qty"),
        sum("f.sale_total_price").alias("total_revenue"),
        count("f.id").alias("sales_count")
    )
    .orderBy(desc("avg_rating"))
)

Запись в ClickHouse

In [13]:
mart_product_sales.write.jdbc(
    url=ch_url,
    table="mart_product_sales",
    mode="overwrite",
    properties=ch_props
)

mart_customer_sales.write.jdbc(
    url=ch_url,
    table="mart_customer_sales",
    mode="overwrite",
    properties=ch_props
)

mart_time_sales.write.jdbc(
    url=ch_url,
    table="mart_time_sales",
    mode="overwrite",
    properties=ch_props
)

mart_store_sales.write.jdbc(
    url=ch_url,
    table="mart_store_sales",
    mode="overwrite",
    properties=ch_props
)

mart_supplier_sales.write.jdbc(
    url=ch_url,
    table="mart_supplier_sales",
    mode="overwrite",
    properties=ch_props
)

mart_product_quality.write.jdbc(
    url=ch_url,
    table="mart_product_quality",
    mode="overwrite",
    properties=ch_props
)